In [ ]:
#!pip install transformers opencv-python pandas

In [28]:
from getpass import getpass

API_KEY = getpass("Introdu cheia OpenRouter: ")

In [ ]:
from pathlib import Path
import base64
import cv2
import pandas as pd
import requests


DATASET_FOLDER = Path("ImageDataset")
OUTPUT_FILE = Path("submission.csv")

MODEL_NAME = "openai/gpt-4.1-mini"

PROMPT = (
    "Identify the main subject of the image and describe it in one concise, factual paragraph of 2–3 sentences. "
    "Include distinctive colors, materials, shapes, context, and any readable text that would help a visual search engine find similar objects or scenes. "
    "Avoid decorative language, unsupported assumptions, and phrases such as 'The image shows.' "
    "Express uncertain identifications cautiously using words such as 'likely,' 'possibly,' or 'appears to be.'"
)


def describe_image(image_path):
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    height, width = image.shape[:2]
    max_dimension = 1600

    scale = min(max_dimension / width, max_dimension / height, 1.0)

    if scale < 1.0:
        new_width = round(width * scale)
        new_height = round(height * scale)

        image = cv2.resize(
            image,
            (new_width, new_height),
            interpolation=cv2.INTER_AREA,
        )

    success, image_buffer = cv2.imencode(
        ".jpg",
        image,
        [cv2.IMWRITE_JPEG_QUALITY, 90],
    )

    image_b64 = base64.b64encode(image_buffer).decode("utf-8")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": PROMPT,
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                "data:image/jpeg;base64,"
                                + image_b64
                            )
                        },
                    },
                ],
            }
        ],
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=120,
    )

    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]


accepted_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}

image_paths = sorted(
    path
    for path in DATASET_FOLDER.iterdir()
    if path.suffix.lower() in accepted_extensions
)

print(f"Found {len(image_paths)} images.")

results = []

for index, image_path in enumerate(image_paths, start=1):
    print(
        f"[{index}/{len(image_paths)}] "
        f"Processing: {image_path.name}"
    )

    try:
        description = describe_image(image_path)

    except Exception as error:
        description = f"ERROR: {error}"

    results.append(
        {
            "image_path": image_path.as_posix(),
            "ai_response": description,
        }
    )

    pd.DataFrame(results).to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig",
    )

print("\nEvaluation completed.")
print(f"Results were saved to: {OUTPUT_FILE}")

Found 16 images.
[1/16] Processing: Apples.jpg
[2/16] Processing: Bicycles_parked.jpg
[3/16] Processing: Blurry_city.jpg
[4/16] Processing: Chair.jpg
[5/16] Processing: Chair2.jpg
[6/16] Processing: Coffee_in_a_mug.jpg
[7/16] Processing: Cute_seal.jpg
[8/16] Processing: Dog_Couch.jpg
[9/16] Processing: Kitchen_interior_design.jpg
[10/16] Processing: Laptop_on_a_desk.jpg
[11/16] Processing: Pedestrians_crossing_the_a_busy_street.jpg
[12/16] Processing: Picture_of_computer_desk.jpg
[13/16] Processing: Riding_a_bike_in_the_park.jpg
[14/16] Processing: Seal.jpg
[15/16] Processing: Traffic_Sign_post.jpg
[16/16] Processing: White_mug.jpg

Evaluation completed.
Results were saved to: submission.csv
